In [1]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

import os

conf = SparkConf()
conf.set('spark.master', 'local[*]')
conf.set('spark.app.name', 'PySpark SQL Basic')

spark = SparkSession.builder\
        .config(conf = conf)\
        .getOrCreate()

spark_home_dir = os.getenv('SPARK_HOME')

### 데이터 프레임 생성

In [2]:
# 파이썬 코드로 생성

data = [{'age': None, 'name': 'Michael'},
        {'age': 30,  'name': 'Andy'},
        {'age': 19, 'name': 'Justin'}]

df = spark.createDataFrame(data)
df.show()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



In [3]:
# 파일 소스로부터 생성

df = spark.read.json(f'{spark_home_dir}/examples/src/main/resources/people.json')
df.show()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



## 데이터 프레임 조작

### DataFrame API

In [4]:
df.printSchema()

root
 |-- age: long (nullable = true)
 |-- name: string (nullable = true)



In [5]:
df.select('name').show()

+-------+
|   name|
+-------+
|Michael|
|   Andy|
| Justin|
+-------+



In [6]:
df.select('age', 'name').show()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



In [7]:
df.filter(df['age'] > 21).show()

+---+----+
|age|name|
+---+----+
| 30|Andy|
+---+----+



In [8]:
df.groupBy('age').count().show()

+----+-----+
| age|count|
+----+-----+
|  19|    1|
|NULL|    1|
|  30|    1|
+----+-----+



### Spark SQL

In [9]:
df.createOrReplaceTempView('df')

In [10]:
spark.sql("""
    SELECT * FROM df
""").show()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



In [11]:
df.createGlobalTempView('df_global')

spark.sql('SELECT * FROM global_temp.df_global').show()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



In [12]:
spark.newSession().sql('SELECT * FROM global_temp.df_global').show()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



### 스키마 지정

In [ ]:
df2 = spark.read.csv(f'{spark_home_dir}/examples/src/main/resources/people.csv')
df2.show()

+------------------+
|               _c0|
+------------------+
|      name;age;job|
|Jorge;30;Developer|
|  Bob;32;Developer|
+------------------+



In [ ]:
df2 = spark.read.csv(f'{spark_home_dir}/examples/src/main/resources/people.csv',
                     sep = ';')
df2.show()

+-----+---+---------+
|  _c0|_c1|      _c2|
+-----+---+---------+
| name|age|      job|
|Jorge| 30|Developer|
|  Bob| 32|Developer|
+-----+---+---------+



In [18]:
df2 = spark.read.csv(f'{spark_home_dir}/examples/src/main/resources/people.csv',
                     sep = ';',
                     header = True)
df2.show()

+-----+---+---------+
| name|age|      job|
+-----+---+---------+
|Jorge| 30|Developer|
|  Bob| 32|Developer|
+-----+---+---------+



In [19]:
df2.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- job: string (nullable = true)



In [20]:
df2 = spark.read.csv(f'{spark_home_dir}/examples/src/main/resources/people.csv',
                     sep = ';',
                     header = True,
                     inferSchema = True)
df2.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)



In [22]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField('name', StringType(), True),
    StructField('age', IntegerType(), True),
    StructField('job', StringType(), True),
])

df2 = spark.read.csv(f'{spark_home_dir}/examples/src/main/resources/people.csv',
                     sep = ';',
                     header = True,
                     schema = schema)
df2.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)



In [15]:
df2 = spark.read.text(f'{spark_home_dir}/examples/src/main/resources/people.txt')
df2.show()

+-----------+
|      value|
+-----------+
|Michael, 29|
|   Andy, 30|
| Justin, 19|
+-----------+



### 데이터 프레임을 파이썬 객체로 가져오기

In [27]:
df_py = df2.collect()

print(df_py)

for i in df_py:
    print(i)

[Row(name='Jorge', age=30, job='Developer'), Row(name='Bob', age=32, job='Developer')]
Row(name='Jorge', age=30, job='Developer')
Row(name='Bob', age=32, job='Developer')


## 데이터 프레임 조작 심화

### 컬럼 추가

In [6]:
df.show()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



In [10]:
import pyspark.sql.functions as f

df.withColumn('age_add_10', f.col('age') + 10)\
    .withColumn('name_startswith', f.substring(f.col('name'), 0, 1)).show()

+----+-------+----------+---------------+
| age|   name|age_add_10|name_startswith|
+----+-------+----------+---------------+
|NULL|Michael|      NULL|              M|
|  30|   Andy|        40|              A|
|  19| Justin|        29|              J|
+----+-------+----------+---------------+



In [11]:
df.withColumns({
    'age_add_10': f.col('age') + 10,
    'name_startswith': f.substring(f.col('name'), 0, 1)
}).show()

+----+-------+----------+---------------+
| age|   name|age_add_10|name_startswith|
+----+-------+----------+---------------+
|NULL|Michael|      NULL|              M|
|  30|   Andy|        40|              A|
|  19| Justin|        29|              J|
+----+-------+----------+---------------+



### 고급 함수

In [20]:
df.withColumn('name_startswith', f.expr("LEFT(name, 1)")).show()

+----+-------+---------------+
| age|   name|name_startswith|
+----+-------+---------------+
|NULL|Michael|              M|
|  30|   Andy|              A|
|  19| Justin|              J|
+----+-------+---------------+



In [23]:
df.select(f.expr('LEFT(name, 1)').alias('name_startswith')).show()

+---------------+
|name_startswith|
+---------------+
|              M|
|              A|
|              J|
+---------------+



In [ ]:
df2 = spark.read.csv('source/survey_results_public.csv',
                     header = True,
                     inferSchema = True)

df2 = df2.select(f.col('LanguageHaveWorkedWith'))

In [11]:
df2.show(5)

+----------------------+
|LanguageHaveWorkedWith|
+----------------------+
|  C++;HTML/CSS;Java...|
|     JavaScript;Python|
|  Assembly;C;Python...|
|  JavaScript;TypeSc...|
|  Bash/Shell;HTML/C...|
+----------------------+
only showing top 5 rows


In [24]:
splitted = df2.withColumn('splitted', f.split(f.trim(f.col('LanguageHaveWorkedWith')), ';'))

splitted.show(5, truncate = False)

+---------------------------------------------+----------------------------------------------------+
|LanguageHaveWorkedWith                       |splitted                                            |
+---------------------------------------------+----------------------------------------------------+
|C++;HTML/CSS;JavaScript;Objective-C;PHP;Swift|[C++, HTML/CSS, JavaScript, Objective-C, PHP, Swift]|
|JavaScript;Python                            |[JavaScript, Python]                                |
|Assembly;C;Python;R;Rust                     |[Assembly, C, Python, R, Rust]                      |
|JavaScript;TypeScript                        |[JavaScript, TypeScript]                            |
|Bash/Shell;HTML/CSS;Python;SQL               |[Bash/Shell, HTML/CSS, Python, SQL]                 |
+---------------------------------------------+----------------------------------------------------+
only showing top 5 rows


In [28]:
splitted.select(f.explode(f.col('splitted'))).show(7)

+-----------+
|        col|
+-----------+
|        C++|
|   HTML/CSS|
| JavaScript|
|Objective-C|
|        PHP|
|      Swift|
| JavaScript|
+-----------+
only showing top 7 rows


### 윈도우 함수

In [3]:
df3 = spark.read.parquet('source/summary.parquet')

df3.show(5)

+---------+----------+-----------+-------------+------------+
|  Country|WeekNumber|NumInvoices|TotalQuantity|InvoiceValue|
+---------+----------+-----------+-------------+------------+
|    Spain|        49|          1|           67|      174.72|
|  Germany|        48|         11|         1795|     3309.75|
|Lithuania|        48|          3|          622|     1598.06|
|  Germany|        49|         12|         1852|     4521.39|
|  Bahrain|        51|          1|           54|      205.74|
+---------+----------+-----------+-------------+------------+
only showing top 5 rows


In [14]:
df3.createOrReplaceTempView('df3')

spark.sql("""
    SELECT *,
          ROW_NUMBER() OVER(PARTITION BY Country ORDER BY TotalQuantity DESC) AS Country_Quantity,
          SUM(InvoiceValue) OVER(PARTITION BY Country ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS SUM1,
          SUM(InvoiceValue) OVER(PARTITION BY Country ROWS BETWEEN 1 PRECEDING AND UNBOUNDED FOLLOWING) AS SUM2
    FROM df3
""").show(5)

+---------+----------+-----------+-------------+------------+----------------+-----------------+------------------+
|  Country|WeekNumber|NumInvoices|TotalQuantity|InvoiceValue|Country_Quantity|             SUM1|              SUM2|
+---------+----------+-----------+-------------+------------+----------------+-----------------+------------------+
|Australia|        49|          1|          214|       258.9|               1|             NULL|1005.0999999999999|
|Australia|        50|          2|          133|      387.95|               2|            258.9|1005.0999999999999|
|Australia|        48|          1|          107|      358.25|               3|646.8499999999999|             746.2|
|  Austria|        50|          2|            3|      257.04|               1|             NULL|            257.04|
|  Bahrain|        51|          1|           54|      205.74|               1|             NULL|            205.74|
+---------+----------+-----------+-------------+------------+-----------

In [19]:
from pyspark.sql import Window

df3.withColumn('Country_Quantity', f.row_number()\
                                    .over(Window.partitionBy(f.col('Country'))\
                                    .orderBy(f.col('TotalQuantity').desc())))\
    .withColumn('Sum1', f.sum(f.col('InvoiceValue'))\
                        .over(Window.partitionBy(f.col('Country'))\
                        .rowsBetween(Window.unboundedPreceding, -1)))\
    .withColumn('Sum2', f.sum(f.col('InvoiceValue'))\
                        .over(Window.partitionBy(f.col('Country'))\
                        .rowsBetween(-1, Window.unboundedFollowing)))\
    .show(5)

+---------+----------+-----------+-------------+------------+----------------+-----------------+------------------+
|  Country|WeekNumber|NumInvoices|TotalQuantity|InvoiceValue|Country_Quantity|             Sum1|              Sum2|
+---------+----------+-----------+-------------+------------+----------------+-----------------+------------------+
|Australia|        49|          1|          214|       258.9|               1|             NULL|1005.0999999999999|
|Australia|        50|          2|          133|      387.95|               2|            258.9|1005.0999999999999|
|Australia|        48|          1|          107|      358.25|               3|646.8499999999999|             746.2|
|  Austria|        50|          2|            3|      257.04|               1|             NULL|            257.04|
|  Bahrain|        51|          1|           54|      205.74|               1|             NULL|            205.74|
+---------+----------+-----------+-------------+------------+-----------

### 정규표현식

In [23]:
df = spark.read.text('source/transfer_cost.txt')

df.show(5, truncate = False)

+---------------------------------------------------------------------------+
|value                                                                      |
+---------------------------------------------------------------------------+
|On 2021-01-04 the cost per ton from 85001 to 85002 is $28.32 at ABC Hauling|
|On 2021-01-04 the cost per ton from 85001 to 85004 is $25.68 at ABC Hauling|
|On 2021-01-04 the cost per ton from 85001 to 85007 is 19.86 at ABC Hauling |
|On 2021-01-04 the cost per ton from 85001 to 85007 is 20.52 at Haul Today  |
|On 2021-01-04 the cost per ton from 85001 to 85010 is 20.72 at Haul Today  |
+---------------------------------------------------------------------------+
only showing top 5 rows


In [29]:
regex_str = r'On (\S+) the cost per ton from (\d+) to (\d+) is (\S+) at (.*)'

df.withColumn('date', f.regexp_extract(f.col('value'), regex_str, 1))\
    .withColumn('departure_zipcode', f.regexp_extract(f.col('value'), regex_str, 2))\
    .withColumn('arrival_zipcode', f.regexp_extract(f.col('value'), regex_str, 3))\
    .withColumn('cost', f.regexp_extract(f.col('value'), regex_str, 4))\
    .withColumn('vendor', f.regexp_extract(f.col('value'), regex_str, 5)).show(5)

+--------------------+----------+-----------------+---------------+------+-----------+
|               value|      date|departure_zipcode|arrival_zipcode|  cost|     vendor|
+--------------------+----------+-----------------+---------------+------+-----------+
|On 2021-01-04 the...|2021-01-04|            85001|          85002|$28.32|ABC Hauling|
|On 2021-01-04 the...|2021-01-04|            85001|          85004|$25.68|ABC Hauling|
|On 2021-01-04 the...|2021-01-04|            85001|          85007| 19.86|ABC Hauling|
|On 2021-01-04 the...|2021-01-04|            85001|          85007| 20.52| Haul Today|
|On 2021-01-04 the...|2021-01-04|            85001|          85010| 20.72| Haul Today|
+--------------------+----------+-----------------+---------------+------+-----------+
only showing top 5 rows
